In [1]:
import pandas as pd
import numpy as np
import glob
import os

print("All imports successful!")

All imports successful!


Creating master dataset

In [2]:
import pandas as pd

dfs = []

for year in range(2000, 2011):
    print(f"Processing {year}")

    file_path = (
        f"C:\\Users\\Niraj Mhatre\\projects\\Mortgage-Portfolio-Risk-Analytics-and-IFRS-9-Provisioning-Framework"
        f"\\Data\\sample_{year}\\sample_orig_{year}.txt"
    )

    df = pd.read_csv(
        file_path,
        sep="|",
        header=None
    )

    df["origination_year"] = year

    dfs.append(df)

master_df = pd.concat(dfs, ignore_index=True)

print(master_df.shape)

Processing 2000
Processing 2001
Processing 2002
Processing 2003
Processing 2004
Processing 2005
Processing 2006
Processing 2007
Processing 2008


C:\Temp\ipykernel_18484\1739466327.py:13: DtypeWarning: Columns (0: 25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


Processing 2009
Processing 2010
(550000, 33)


In [3]:
column_names = ["cred_score", "first_payment_date", "first_time_homebuyer_flag", "maturity_date", 
                "metro_code_msa", "mortgage_insurance_percent", "unit_no", "occupancy_status",
                "og_cltv", "og_dti", "og_upb", "og_ltv",
                "og_int_rate", "channel", "ppm_flag", "amort_type",
                "prop_state", "prop_type", "postal_code", "loan_seq_no",
                "loan_purpose", "og_loan_term", "no_of_borrowers", "seller_name",
                "servicer_name", "super_conforming_flag", "pre_relief_refinance_loan_seq_no", "special_elig_program",
                "relief_refinance_indicator", "prop_val_method", "int_only_indicator", "MI_cal_indicator"
                ]

In [4]:
final_column_names = column_names + ['origination_year']
master_df.columns = final_column_names

In [5]:
master_df["loan_seq_no"].nunique()

550000

In [6]:
master_df.head()

,cred_score,first_payment_date,first_time_homebuyer_flag,maturity_date,metro_code_msa,mortgage_insurance_percent,unit_no,occupancy_status,og_cltv,og_dti,...,seller_name,servicer_name,super_conforming_flag,pre_relief_refinance_loan_seq_no,special_elig_program,relief_refinance_indicator,prop_val_method,int_only_indicator,MI_cal_indicator,origination_year
0,790,200211,N,203003,NaN,30,1,P,92,29,...,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,9,2000
1,771,200011,N,201510,NaN,0,1,P,61,30,...,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,9,2000
2,762,200110,N,203003,39300.0,0,3,P,76,23,...,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,9,2000
3,737,200104,N,203004,16974.0,25,1,P,87,35,...,"NORWEST MORTGAGE, INC.","WELLS FARGO HOME MORTGAGE, INC.",NaN,NaN,9,NaN,7,N,9,2000
4,594,200003,N,203002,23460.0,0,1,P,80,24,...,Other sellers,CHASE MANHATTAN MORTGAGE CORPORATION,NaN,NaN,9,NaN,7,N,9,2000


The objective here to check if svc{year}.csv has the data of that loan for all consequent years

In [7]:
import pandas as pd
# Define column names (Freddie Mac standard svcg layout)
svcg_cols = [
    'loan_seq_no', 'monthly_period', 'current_upb', 'current_loan_delinquency_status',
    'loan_age', 'remaining_months_to_legal_maturity', 'defect_settlement_date',
    'modifications_flag', 'zero_balance_code', 'zero_balance_effective_date',
    'current_interest_rate', 'current_deferred_upb', 'due_date_last_paid_installment',
    'mi_recoveries', 'net_sales_proceeds', 'non_mi_recoveries', 'expenses',
    'legal_costs', 'maintenance_preservation_costs', 'taxes_insurance',
    'misc_expenses', 'actual_loss', 'modification_cost', 'step_modification_flag',
    'deferred_payment_modification', 'estimated_ltv', 'zero_balance_removal_upb',
    'delinquent_accrued_interest', 'delinquency_due_to_disaster',
    'borrower_assistance_status_code', 'current_period_modification_loss_amount',
    'interest_bearing_upb'
]

# Load full svcg_2003 file
svcg = pd.read_csv("C:\\Users\\Niraj Mhatre\\projects\\Mortgage-Portfolio-Risk-Analytics-and-IFRS-9-Provisioning-Framework\\Data\\sample_2003\\sample_svcg_2003.txt", sep='|', header=None, names=svcg_cols, dtype={'loan_seq_no': str, 'monthly_period': str})

#  What's the full date range?
print("Monthly period range:")
print(f"  Earliest: {svcg['monthly_period'].min()}")
print(f"  Latest:   {svcg['monthly_period'].max()}")

#  How many unique loans are tracked?
print(f"\nTotal unique loans: {svcg['loan_seq_no'].nunique():,}")
print(f"Total monthly records: {len(svcg):,}")

#  Records from 2004 onwards (loans still alive after origination year)
svcg_post2003 = svcg[svcg['monthly_period'] >= '200401']
print(f"\nRecords from Jan 2004 onwards: {len(svcg_post2003):,}")
print(f"Unique loans still active in 2004+: {svcg_post2003['loan_seq_no'].nunique():,}")

#  Year-wise breakdown of records
svcg['year'] = svcg['monthly_period'].str[:4]
print("\nRecords per year:")
print(svcg['year'].value_counts().sort_index())

# --- CHECK 5: Loans still active by year (unique loan count per year)


C:\Temp\ipykernel_18484\3637004282.py:18: DtypeWarning: Columns (0: current_loan_delinquency_status, 1: modifications_flag, 2: step_modification_flag, 3: deferred_payment_modification, 4: delinquency_due_to_disaster, 5: borrower_assistance_status_code) have mixed types. Specify dtype option on import or set low_memory=False.
  svcg = pd.read_csv("C:\\Users\\Niraj Mhatre\\projects\\Mortgage-Portfolio-Risk-Analytics-and-IFRS-9-Provisioning-Framework\\Data\\sample_2003\\sample_svcg_2003.txt", sep='|', header=None, names=svcg_cols, dtype={'loan_seq_no': str, 'monthly_period': str})


Monthly period range:
  Earliest: 200301
  Latest:   202509

Total unique loans: 50,000
Total monthly records: 4,073,882

Records from Jan 2004 onwards: 3,818,439
Unique loans still active in 2004+: 48,339

Records per year:
year
2003    255443
2004    543051
2005    482995
2006    428909
2007    391596
2008    361409
2009    323416
2010    277027
2011    224769
2012    179870
2013    134704
2014    107957
2015     89730
2016     72976
2017     56750
2018     36997
2019     24314
2020     21036
2021     17330
2022     14329
2023     12012
2024     10242
2025      7020
Name: count, dtype: int64


In [8]:
print("\nUnique active loans per year:")
print(svcg.groupby('year')['loan_seq_no'].nunique().sort_index())


Unique active loans per year:
year
2003    44376
2004    47839
2005    42815
2006    37382
2007    33982
2008    31334
2009    29045
2010    24764
2011    20422
2012    16701
2013    12904
2014     9747
2015     8146
2016     6711
2017     5329
2018     4018
2019     2163
2020     1867
2021     1582
2022     1289
2023     1100
2024      899
2025      807
Name: loan_seq_no, dtype: int64


Since the dataset has passed this sanity check we combine a master dataset for svc/process files

In [9]:
import pandas as pd

dfs = []

for year in range(2000, 2011):
    print(f"Processing {year}")

    file_path = (
        f"C:\\Users\\Niraj Mhatre\\projects\\Mortgage-Portfolio-Risk-Analytics-and-IFRS-9-Provisioning-Framework\\Data\\sample_{year}\\sample_svcg_{year}.txt"
    )

    df = pd.read_csv(
        file_path,
        sep="|",
        header=None
    )

    df["origination_year"] = year

    dfs.append(df)

master_monthly_df = pd.concat(dfs, ignore_index=True)

print(master_monthly_df.shape)

Processing 2000


C:\Temp\ipykernel_18484\2519369399.py:12: DtypeWarning: Columns (0: 3, 1: 7, 2: 23, 3: 24, 4: 28, 5: 29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


Processing 2001
Processing 2002
Processing 2003
Processing 2004
Processing 2005
Processing 2006


C:\Temp\ipykernel_18484\2519369399.py:12: DtypeWarning: Columns (0: 3, 1: 7, 2: 23, 3: 24, 4: 28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


Processing 2007


C:\Temp\ipykernel_18484\2519369399.py:12: DtypeWarning: Columns (0: 24, 1: 28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


Processing 2008
Processing 2009
Processing 2010
(33058391, 33)


renaming cols in monthly data

In [10]:
master_monthly_df.head()

,0,1,2,3,4,5,6,7,8,9,...,23,24,25,26,27,28,29,30,31,origination_year
0,F00Q10000035,200210,101000.0,0,0,329.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,101000.0,2000
1,F00Q10000035,200211,101000.0,0,1,328.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,101000.0,2000
2,F00Q10000035,200212,101000.0,0,2,327.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,101000.0,2000
3,F00Q10000035,200301,101000.0,0,3,326.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,101000.0,2000
4,F00Q10000035,200302,100000.0,0,4,325.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100000.0,2000


In [11]:
master_monthly_df.shape

(33058391, 33)

In [12]:
monthly_cols = [
    "loan_seq_no",
    "monthly_reporting_period",
    "current_actual_upb",
    "current_loan_delinquency_status",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "defect_settlement_date",
    "modification_flag",
    "zero_balance_code",
    "zero_balance_effective_date",
    "current_interest_rate",
    "current_non_interest_bearing_upb",
    "ddlpi",
    "mi_recoveries",
    "net_sale_proceeds",
    "non_mi_recoveries",
    "total_expenses",
    "legal_costs",
    "maintenance_and_preservation_costs",
    "taxes_and_insurance",
    "miscellaneous_expenses",
    "actual_loss_calculation",
    "cumulative_modification_cost",
    "interest_rate_step_indicator",
    "payment_deferral_flag",
    "eltv",
    "zero_balance_removal_upb",
    "delinquent_accrued_interest",
    "delinquency_due_to_disaster",
    "borrower_assistance_status_code",
    "current_month_modification_cost",
    "interest_bearing_upb",
    "origination_year"
]

In [13]:
len(monthly_cols)

33

In [14]:
master_monthly_df.columns = monthly_cols

check for null values

In [15]:

na_val = master_df.isnull().sum()
print(na_val[na_val > 0] / len(master_df))

metro_code_msa                      0.315813
super_conforming_flag               0.996125
pre_relief_refinance_loan_seq_no    0.963627
relief_refinance_indicator          0.963627
dtype: float64


imputing the values in nessecary ways

In [16]:
master_df[["metro_code_msa", "super_conforming_flag", "pre_relief_refinance_loan_seq_no", "relief_refinance_indicator"]]

,metro_code_msa,super_conforming_flag,pre_relief_refinance_loan_seq_no,relief_refinance_indicator
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,39300.0,NaN,NaN,NaN
3,16974.0,NaN,NaN,NaN
4,23460.0,NaN,NaN,NaN
...,...,...,...,...
549995,13900.0,NaN,NaN,NaN
549996,24220.0,NaN,NaN,NaN
549997,15500.0,NaN,NaN,NaN
549998,20500.0,NaN,NaN,NaN


In [17]:
#checkimg the unique values in the columns with missing values
np.where(
    master_df["pre_relief_refinance_loan_seq_no"].notna()
)

(array([459038, 459682, 459683, ..., 549843, 549864, 549866],
       shape=(20005,)),)

In [18]:
master_df['super_conforming_flag'] = master_df['super_conforming_flag'].fillna('N')
# N Because a blank space in this dataset explicitly means "No / Standard Conforming" so we will have to use it in different ways later

In [19]:
master_df[["metro_code_msa", "pre_relief_refinance_loan_seq_no"]] = master_df[
    ["metro_code_msa", "pre_relief_refinance_loan_seq_no"]
].fillna("Unknown")

In [20]:
# Properly fill the NaNs by reassigning the column
master_df["relief_refinance_indicator"] = master_df[
    "relief_refinance_indicator"
].fillna("not applicable")

indices = np.where(master_df["relief_refinance_indicator"] == "not applicable")
print(indices)


(array([     0,      1,      2, ..., 549997, 549998, 549999],
      shape=(529995,)),)


# Imputing values for monthly data files

### first we check the prop of loans defaulted as it will help us decide what to do with the missing values

zero balance code is the code which tells if the loan is inactive along with it's reason,

🔴 Defaulted / Credit Events (Involuntary Termination)
If a loan's balance drops to zero because of a default, foreclosure, or liquidation, it will be flagged with one of these codes:

02 = Third Party Sale: The property was sold to a third party at a foreclosure auction.

03 = Short Sale or Charge Off: The property was sold for less than the remaining loan balance, or the servicer wrote off the remaining debt as a loss.

09 = REO Disposition: The foreclosure completed, Freddie Mac took ownership of the property (Real Estate Owned), and this code triggers when they finally sell/dispose of the property.

🟢 Successfully Paid Off (Voluntary Termination)
01 = Prepaid or Matured: The borrower successfully paid off the loan in full (e.g., through a refinancing, selling the home, or making the final scheduled payment).

ℹ️ Other Structural Zero Balance Codes
For completeness, the file also includes a few codes that don't neatly fall into a simple "borrower paid vs. borrower defaulted" category, usually representing administrative or portfolio changes:

15 = Whole Loan Sales: The loan was sold out of the pool to another investor.

16 = Reperforming Loan Securitizations: The loan previously delinquent became clean again and was repackaged into a new security.

96 = Defect Prior to Other Termination Event: The loan was removed from the dataset due to an eligibility or underwriting defect identified by the seller/servicer before any final payoff or default happened.

In [21]:
# looking at proportion of each based on if they are defaulted or not or some other situation we have 3 classes here
print(master_monthly_df["current_loan_delinquency_status"].value_counts(normalize=True
        ))

current_loan_delinquency_status
0      8.360614e-01
0      1.195402e-01
1      1.659385e-02
2      5.276512e-03
RA     2.767406e-03
           ...     
198    3.024951e-08
199    3.024951e-08
200    3.024951e-08
201    3.024951e-08
202    3.024951e-08
Name: proportion, Length: 388, dtype: float64


In [22]:
arr = master_monthly_df.loc[
    master_monthly_df["zero_balance_code"].notna(),
    "zero_balance_code"
].value_counts()
# adding query of notna else the value counts will also count the NaNs which is not what we want here
print(arr)
type(arr)

zero_balance_code
1.0     511190
9.0      10878
16.0      4646
3.0       4178
2.0       2712
96.0      2145
15.0      1000
Name: count, dtype: int64


pandas.Series

In [23]:
proportion_default = (arr[2]+arr[3]+arr[9]+ arr[15]) / arr.sum()
proportion_paid = arr[1] / arr.sum()
proportion_undecided = (arr[16] + arr[96] ) / arr.sum()

print("default:",proportion_default,"paid:",proportion_paid,"undecided:",proportion_undecided)

default: 0.034966064212509015 paid: 0.9523818395562916 undecided: 0.012652096231199313


since default rate is only 0.03 these values are precious for us

In [24]:
na_val = master_monthly_df.isnull().sum()
print(na_val[na_val > 0] / len(master_monthly_df))

remaining_months_to_legal_maturity    0.000009
defect_settlement_date                0.999892
modification_flag                     0.971975
zero_balance_code                     0.983764
zero_balance_effective_date           0.983764
ddlpi                                 0.998187
mi_recoveries                         0.999477
net_sale_proceeds                     0.999475
non_mi_recoveries                     0.999477
total_expenses                        0.999477
legal_costs                           0.999477
maintenance_and_preservation_costs    0.999477
taxes_and_insurance                   0.999477
miscellaneous_expenses                0.999477
actual_loss_calculation               0.999475
cumulative_modification_cost          0.999655
interest_rate_step_indicator          0.971975
payment_deferral_flag                 0.998118
eltv                                  0.903336
zero_balance_removal_upb              0.983764
delinquent_accrued_interest           0.999475
delinquency_d

In [25]:
zero_na = master_monthly_df["zero_balance_code"].isna()
not_na = master_monthly_df["zero_balance_code"].notna()
na_data = master_monthly_df[zero_na]
not_na_data = master_monthly_df[not_na]

In [26]:
not_na_data["loan_seq_no"].nunique()

536749

In [27]:
master_monthly_df["loan_seq_no"].unique()

<ArrowStringArray>
['F00Q10000035', 'F00Q10000049', 'F00Q10000054', 'F00Q10000090',
 'F00Q10000095', 'F00Q10000106', 'F00Q10000108', 'F00Q10000120',
 'F00Q10000184', 'F00Q10000193',
 ...
 'F10Q40595093', 'F10Q40595106', 'F10Q40595111', 'F10Q40595182',
 'F10Q40595246', 'F10Q40595335', 'F10Q40595352', 'F10Q40595382',
 'F10Q40595385', 'F10Q40595389']
Length: 550000, dtype: str

In [28]:
master_monthly_df[master_monthly_df["loan_seq_no"] == "F00Q10012762"][["monthly_reporting_period", "zero_balance_code"]]

,monthly_reporting_period,zero_balance_code
26839,200002,NaN
26840,200003,NaN
26841,200004,NaN
26842,200005,NaN
26843,200006,NaN
...,...,...
27142,202505,NaN
27143,202506,NaN
27144,202507,NaN
27145,202508,NaN


for sanity check, we check weather the loans which havent been matured yet dont have val

In [29]:
master_monthly_df.monthly_reporting_period

0           200210
1           200211
2           200212
3           200301
4           200302
             ...  
33058386    202002
33058387    202003
33058388    202004
33058389    202005
33058390    202006
Name: monthly_reporting_period, Length: 33058391, dtype: int64

In [30]:
######################################################################################
mask = master_monthly_df["monthly_reporting_period"].astype(str).str[:4] == "2025"
master_monthly_df.loc[mask, ["loan_seq_no", "zero_balance_code"]]

,loan_seq_no,zero_balance_code
27138,F00Q10012762,NaN
27139,F00Q10012762,NaN
27140,F00Q10012762,NaN
27141,F00Q10012762,NaN
27142,F00Q10012762,NaN
...,...,...
33058119,F10Q40595246,NaN
33058120,F10Q40595246,NaN
33058121,F10Q40595246,NaN
33058122,F10Q40595246,NaN


here we understand why there are null values in zero_balance_code and the remaining cols are not to be touched as the loans are still going on and in that case same goes for zero_balance_effective_date 

#### we work in "remaining_months_to_legal_maturity" col

In [31]:
remaining_months_notnull = np.where(master_monthly_df["remaining_months_to_legal_maturity"].notnull())

In [32]:
master_monthly_df["remaining_months_to_legal_maturity"].isna().sum()

np.int64(304)

In [55]:
kk =master_monthly_df[master_monthly_df["remaining_months_to_legal_maturity"].isna()]
kk.nunique()

loan_seq_no                             5
monthly_reporting_period              170
current_actual_upb                    238
current_loan_delinquency_status        25
loan_age                              170
remaining_months_to_legal_maturity      0
defect_settlement_date                  0
modification_flag                       1
zero_balance_code                       0
zero_balance_effective_date             0
current_interest_rate                   5
current_non_interest_bearing_upb        1
ddlpi                                   0
mi_recoveries                           0
net_sale_proceeds                       0
non_mi_recoveries                       0
total_expenses                          0
legal_costs                             0
maintenance_and_preservation_costs      0
taxes_and_insurance                     0
miscellaneous_expenses                  0
actual_loss_calculation                 0
cumulative_modification_cost            0
interest_rate_step_indicator      

since the proportion is very small he idea is to drop these rows if they dont contain much info, i.e. we will deal with rows according to the situation in next models

the same goes with defect settlement date and zero balance effective date which are also only applicable for a subset of the data. So we will have to handle these columns in a similar way as we did with the zero balance code column. We can create separate datasets for the rows where these columns are not null and analyze them separately.

In [33]:
# Fill NaNs with a string descriptor if keeping as text
master_monthly_df["modification_flag"] = master_monthly_df["modification_flag"].fillna("N")

all the missing cols with same values as mi_recovery
are to be imputed this way while irfs framework
Isolate credit event loans to calculate historical loss severity
defaulted_portfolio = master_monthly_df[master_monthly_df["ZERO BALANCE CODE"].isin(["02", "03", "09"])].copy()

For these rows, fill any accidental sub-NaNs with 0 so the math doesn't break
loss_cols = ["total_expenses", "legal_costs", "maintenance_and_preservation_costs", 
             "taxes_and_insurance", "miscellaneous_expenses", "mi_recoveries", "non_mi_recoveries"]
defaulted_portfolio[loss_cols] = defaulted_portfolio[loss_cols].fillna(0)

for prob calculation we can simply drop them

and the same goes for col net sales proceeds

# Create a running indicator of historical modification costs for active loans
#master_monthly_df["had_modification_costs"] = (master_monthly_df["cumulative_modification_cost"].fillna(0) > 0).astype(int)

remaining_months_to_legal_maturity    0.000009
defect_settlement_date                0.999892
modification_flag                     0.971975
zero_balance_code                     0.983764
zero_balance_effective_date           0.983764
ddlpi                                 0.998187
mi_recoveries                         0.999477
net_sale_proceeds                     0.999475
non_mi_recoveries                     0.999477
total_expenses                        0.999477
legal_costs                           0.999477
maintenance_and_preservation_costs    0.999477
taxes_and_insurance                   0.999477
miscellaneous_expenses                0.999477
actual_loss_calculation               0.999475
cumulative_modification_cost          0.999655
interest_rate_step_indicator          0.971975
payment_deferral_flag                 0.998118
eltv                                  0.903336
zero_balance_removal_upb              0.983764
delinquent_accrued_interest           0.999475
delinquency_due_to_disaster           0.998891
borrower_assistance_status_code       0.998154
current_month_modification_cost       0.970576

In [34]:
arr = master_monthly_df.interest_rate_step_indicator.notna()

In [35]:
master_monthly_df[arr].interest_rate_step_indicator

19996       N
19997       N
19998       N
19999       N
20000       N
           ..
33054942    N
33054943    N
33054944    N
33054945    N
33054946    N
Name: interest_rate_step_indicator, Length: 926467, dtype: str

In [36]:
# interest step indicator should be done the same way we did for modification flag
master_monthly_df["modification_flag"] = master_monthly_df["modification_flag"].fillna("unknown")

In [37]:
# Impute NaNs with 'N' to represent "No Deferral"
master_monthly_df["payment_deferral_flag"] = master_monthly_df["payment_deferral_flag"].fillna("N")

In [38]:
# Convert NaNs to 0 on the fly for portfolio-wide loss calculations
gross_exposure_at_termination = master_monthly_df["zero_balance_removal_upb"].fillna(0)

In [39]:
# Isolate the total exposure lost strictly due to credit defaults
default_codes = ["02", "03", "09", "15"]
defaulted_exposure = master_monthly_df[
    (master_monthly_df["zero_balance_removal_upb"].fillna(0) > 0) &
    (master_monthly_df["zero_balance_code"].isin(default_codes))
]["zero_balance_removal_upb"]

remaining NaNs should be treated with the same idea later on according to problem statement

#### Saving files

In [ ]:
import os
import pandas as pd

# 1. List all columns that contain codes, flags, or postal strings that can get mixed with NaNs
mixed_string_cols = [
    "metro_code_msa", 
    "postal_code", 
    "current_loan_delinquency_status"
]

# 2. Safely cast them to pandas' official nullable 'string' type
for col in mixed_string_cols:
    if col in master_df.columns:
        master_df[col] = master_df[col].astype(str).replace(['nan', 'NaN', 'None', '<NA>'], None).astype("string")
        
    if col in master_monthly_df.columns:
        master_monthly_df[col] = master_monthly_df[col].astype(str).replace(['nan', 'NaN', 'None', '<NA>'], None).astype("string")

# -------------------------------------------------------------------------
# Save as Parquet (Now it will run smoothly!)
# -------------------------------------------------------------------------
master_df.to_parquet(os.path.join(output_dir, "master_origination_clean.parquet"), index=False)
master_monthly_df.to_parquet(os.path.join(output_dir, "master_monthly_performance_clean.parquet"), index=False)

print("Files successfully saved as Parquet format!")

In [ ]:
import os
import pandas as pd

# -------------------------------------------------------------------------
# 1. Enforce String Extension Type on Mixed Identification Columns
# -------------------------------------------------------------------------
# PyArrow requires uniform data types. We convert columns that mix strings 
# and float NaNs into pandas' official nullable 'string' type.

orig_mixed_cols = ["metro_code_msa", "postal_code", "seller_name", "servicer_name"]
monthly_mixed_cols = ["current_loan_delinquency_status", "zero_balance_code"]

# Clean Origination DataFrame
for col in orig_mixed_cols:
    if col in master_df.columns:
        master_df[col] = (
            master_df[col]
            .astype(str)
            .replace(["nan", "NaN", "None", "<NA>"], None)
            .astype("string")
        )

# Clean Monthly Performance DataFrame
for col in monthly_mixed_cols:
    if col in master_monthly_df.columns:
        master_monthly_df[col] = (
            master_monthly_df[col]
            .astype(str)
            .replace(["nan", "NaN", "None", "<NA>"], None)
            .astype("string")
        )

# -------------------------------------------------------------------------
# 2. Save as Parquet (Highly Compressed & Safe Data Types)
# -------------------------------------------------------------------------
orig_output_path = os.path.join(output_dir, "master_origination_clean.parquet")
monthly_output_path = os.path.join(output_dir, "master_monthly_performance_clean.parquet")

# Writing files to disk
master_df.to_parquet(orig_output_path, index=False, engine="pyarrow")
master_monthly_df.to_parquet(monthly_output_path, index=False, engine="pyarrow")

print("=" * 50)
print("🎉 SUCCESS: Files successfully saved as Parquet format!")
print(f" -> Origination Saved: {orig_output_path}")
print(f" -> Performance Saved: {monthly_output_path}")
print("=" * 50)